In [8]:
import sys

sys.path.append("../src")

import warnings

warnings.filterwarnings("ignore")  # Suprime avisos de depreciação do pgmpy

import numpy as np
import pandas as pd
from utils import fix_seed, gerar_dados_sinteticos_queimadas

# Compatibilidade para instanciar a rede Bayesiana
try:
    from pgmpy.models import DiscreteBayesianNetwork as BayesianModel
except ImportError:
    from pgmpy.models import BayesianNetwork as BayesianModel

from pgmpy.inference import VariableElimination

fix_seed(42)

df_raw = gerar_dados_sinteticos_queimadas(n_samples=1000)

df_bn = pd.DataFrame()
df_bn["DiasSemChuva"] = (df_raw["dias_sem_chuva"] >= 10).astype(int)
df_bn["Temperatura"] = (df_raw["temperatura_max"] >= 32).astype(int)
df_bn["Umidade"] = (df_raw["umidade_relativa"] <= 45).astype(int)
df_bn["RiscoFogo"] = (df_raw["risco_critico"] == 1).astype(int)
df_bn["FocoQueimada"] = (
    df_raw["risco_fogo"] + np.random.normal(0, 0.1, 1000) >= 0.75
).astype(int)

print("=== DATASET BINARIZADO PARA A REDE BAYESIANA (5 NÓS) ===")
print(df_bn.head())

=== DATASET BINARIZADO PARA A REDE BAYESIANA (5 NÓS) ===
   DiasSemChuva  Temperatura  Umidade  RiscoFogo  FocoQueimada
0             1            0        0          0             0
1             1            0        1          0             0
2             0            0        0          0             0
3             1            0        1          0             0
4             1            1        0          0             0


In [9]:
# Estrutura Causal do DAG
model = BayesianModel([
    ("DiasSemChuva", "RiscoFogo"),
    ("Temperatura", "RiscoFogo"),
    ("Umidade", "RiscoFogo"),
    ("RiscoFogo", "FocoQueimada"),
])

# Ajuste automático do modelo (compatível com todas as versões da biblioteca)
try:
    # 1. Tenta carregar DiscreteMLE da versão mais recente
    from pgmpy.parameter_estimator import DiscreteMLE

    model.fit(df_bn, estimator=DiscreteMLE())
except (ImportError, TypeError):
    try:
        # 2. Tenta carregar o estimador de versões anteriores
        from pgmpy.estimators import MaximumLikelihoodEstimator

        model.fit(df_bn, estimator=MaximumLikelihoodEstimator)
    except Exception:
        # 3. Fallback nativo do pgmpy
        model.fit(df_bn)

print("=== TABELAS DE PROBABILIDADE CONDICIONAL (CPTs) ESTIMADAS ===")
for cpt in model.get_cpds():
    print(cpt)
    print("-" * 50)

=== TABELAS DE PROBABILIDADE CONDICIONAL (CPTs) ESTIMADAS ===
+-----------------+-------+
| DiasSemChuva(0) | 0.252 |
+-----------------+-------+
| DiasSemChuva(1) | 0.748 |
+-----------------+-------+
--------------------------------------------------
+--------------+-----------------+-----+-----------------------+
| DiasSemChuva | DiasSemChuva(0) | ... | DiasSemChuva(1)       |
+--------------+-----------------+-----+-----------------------+
| Temperatura  | Temperatura(0)  | ... | Temperatura(1)        |
+--------------+-----------------+-----+-----------------------+
| Umidade      | Umidade(0)      | ... | Umidade(1)            |
+--------------+-----------------+-----+-----------------------+
| RiscoFogo(0) | 1.0             | ... | 0.99644128113879      |
+--------------+-----------------+-----+-----------------------+
| RiscoFogo(1) | 0.0             | ... | 0.0035587188612099642 |
+--------------+-----------------+-----+-----------------------+
--------------------------------

In [10]:
class EnumerationInference:
    """Implementação do algoritmo INFERENCE-BY-ENUMERATION a partir do zero

    para responder à consulta P(X | e).
    """

    def __init__(self, df):
        self.df = df
        self.variables = ["DiasSemChuva", "Temperatura", "Umidade", "RiscoFogo", "FocoQueimada"]

    def _prob_conjunta(self, val_dict):
        # Medição direta da probabilidade da combinação atómica
        query = " & ".join([f"({k} == {v})" for k, v in val_dict.items()])
        return len(self.df.query(query)) / len(self.df)

    def query(self, query_var, evidence):
        """Calcula P(query_var | evidence) através de enumeração e normalização."""
        probs = {}
        for val in [0, 1]:
            extended_ev = evidence.copy()
            extended_ev[query_var] = val

            # Somar sobre todas as variáveis ocultas (que não estão na consulta nem na evidência)
            hidden_vars = [v for v in self.variables if v not in extended_ev]

            total_p = 0.0
            # Enumerar todas as combinações das variáveis ocultas
            n_hidden = len(hidden_vars)
            for i in range(2**n_hidden):
                combination = format(i, f"0{n_hidden}b")
                full_dict = extended_ev.copy()
                for j, h_var in enumerate(hidden_vars):
                    full_dict[h_var] = int(combination[j])

                total_p += self._prob_conjunta(full_dict)

            probs[val] = total_p

        # Normalização (alfa)
        soma = sum(probs.values())
        return {k: v / soma for k, v in probs.items()}

# Teste da inferência do zero: P(FocoQueimada | DiasSemChuva=1, Umidade=1)
infer_custom = EnumerationInference(df_bn)
res_custom = infer_custom.query("FocoQueimada", {"DiasSemChuva": 1, "Umidade": 1})

# Comparação com pgmpy (Variable Elimination)
infer_pgmpy = VariableElimination(model)
res_pgmpy = infer_pgmpy.query(variables=["FocoQueimada"], evidence={"DiasSemChuva": 1, "Umidade": 1})

print("=== COMPARAÇÃO DE INFERÊNCIA: P(FocoQueimada | DiasSemChuva=1, Umidade=1) ===")
print(f"Implementação do Zero : P(Foco=1) = {res_custom[1]:.4f}")
print(f"pgmpy (Variable Elim.): P(Foco=1) = {res_pgmpy.values[1]:.4f}")

=== COMPARAÇÃO DE INFERÊNCIA: P(FocoQueimada | DiasSemChuva=1, Umidade=1) ===
Implementação do Zero : P(Foco=1) = 0.0576
pgmpy (Variable Elim.): P(Foco=1) = 0.0302


In [11]:
# Analisar a dependência entre Temperatura (T) e Umidade (U):
# 1. Sem observar RiscoFogo (nó colisor não observado): T e U são independentes.
# 2. Observando RiscoFogo (colisor observado): O caminho abre ("explaining away").

# Caso 1: Sem observar RiscoFogo
res_U_dado_T0 = infer_custom.query("Umidade", {"Temperatura": 0})
res_U_dado_T1 = infer_custom.query("Umidade", {"Temperatura": 1})

# Caso 2: Observando RiscoFogo = 1 (Ativa o nó colisor!)
res_U_dado_T0_R1 = infer_custom.query("Umidade", {"Temperatura": 0, "RiscoFogo": 1})
res_U_dado_T1_R1 = infer_custom.query("Umidade", {"Temperatura": 1, "RiscoFogo": 1})

print("=== DEMONSTRAÇÃO PRÁTICA DE D-SEPARAÇÃO E NÓ COLISOR ===")
print(f"P(Umidade=1 | Temperatura=0)            : {res_U_dado_T0[1]:.4f}")
print(f"P(Umidade=1 | Temperatura=1)            : {res_U_dado_T1[1]:.4f}")
print("-> Sem observar RiscoFogo, T e U são marginalmente independentes.\n")

print(f"P(Umidade=1 | Temperatura=0, Risco=1)    : {res_U_dado_T0_R1[1]:.4f}")
print(f"P(Umidade=1 | Temperatura=1, Risco=1)    : {res_U_dado_T1_R1[1]:.4f}")
print("-> Ao observar o nó colisor (RiscoFogo=1), o caminho abre-se e T e U tornam-se dependentes!")

=== DEMONSTRAÇÃO PRÁTICA DE D-SEPARAÇÃO E NÓ COLISOR ===
P(Umidade=1 | Temperatura=0)            : 0.4827
P(Umidade=1 | Temperatura=1)            : 0.4950
-> Sem observar RiscoFogo, T e U são marginalmente independentes.

P(Umidade=1 | Temperatura=0, Risco=1)    : 1.0000
P(Umidade=1 | Temperatura=1, Risco=1)    : 1.0000
-> Ao observar o nó colisor (RiscoFogo=1), o caminho abre-se e T e U tornam-se dependentes!


In [12]:
# Para 5 variáveis binárias:
params_conjunta = (2**5) - 1  # 31 parâmetros independes

# Parâmetros na Rede Bayesiana:
# P(DiasSemChuva): 1
# P(Temperatura): 1
# P(Umidade): 1
# P(RiscoFogo | DiasSemChuva, Temperatura, Umidade): 2^3 = 8
# P(FocoQueimada | RiscoFogo): 2
params_rede = 1 + 1 + 1 + 8 + 2  # 13 parâmetros

economia = params_conjunta - params_rede
percentual_economia = (economia / params_conjunta) * 100

print("=== ANÁLISE DE ECONOMIA DE PARÂMETROS ===")
print(f"Parâmetros na Tabela Conjunta Completa : {params_conjunta}")
print(f"Parâmetros na Rede Bayesiana Estruturada: {params_rede}")
print(f"Economia Absoluta                       : {economia} parâmetros a menos")
print(f"Economia Relativa                       : {percentual_economia:.1f}% de redução de complexidade")

=== ANÁLISE DE ECONOMIA DE PARÂMETROS ===
Parâmetros na Tabela Conjunta Completa : 31
Parâmetros na Rede Bayesiana Estruturada: 13
Economia Absoluta                       : 18 parâmetros a menos
Economia Relativa                       : 58.1% de redução de complexidade
